In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import scvelo as scv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- Color scheme (same as before) ---
labels = np.asarray(adata.obs["state_info"].values)
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]

cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"

# --- Genes to plot ---
genes_to_plot = ["Ltf", "Mmp12", "Tph1", "Ltbp1"]

# --- Reproducible subsample ---
np.random.seed(42)
n_cells = adata.n_obs
idx = np.random.choice(n_cells, min(6000, n_cells), replace=False)
adata_sub = adata[idx].copy()

# --- Plot phase portraits ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, gene in zip(axes, genes_to_plot):
    scv.pl.scatter(
        adata_sub,
        basis=gene,
        vkey="velocity",
        color="state_info",
        palette=colmap,
        linewidth=2.5,
        frameon=False,
        size=400,
        alpha=0.4,
        legend_loc="none",
        fontsize=32,
        title=gene,
        ax=ax,
        show=False,
    )

    # remove any residual steady-state legend
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

    # steady-state line
    if "velocity_gamma" in adata.var.columns:
        gamma = adata.var.loc[gene, "velocity_gamma"]
        if np.isfinite(gamma):
            Mu_gene = adata_sub.layers["Mu"][:, adata.var_names == gene]
            if hasattr(Mu_gene, "toarray"):
                Mu_gene = Mu_gene.toarray().ravel()
            else:
                Mu_gene = np.ravel(Mu_gene)
            s_fit = np.linspace(0, np.max(Mu_gene), 200)
            u_fit = s_fit / gamma
            ax.plot(u_fit, s_fit, "--", color="orange", lw=2.5, alpha=0.9)

    ax.set_xlabel("Unspliced", fontsize=24)
    ax.set_ylabel("Spliced", fontsize=24)
    ax.grid(False)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()